# Phase 4 — ML Modeling

Random Forest + XGBoost trained on `phase3_feature_table.csv`, evaluated with **Spatial Cluster K-Fold**
(KMeans on projected coordinates), per the semivariogram finding in Phase 2/3: the `block_id` grid is not
a valid spatial-independence unit at this sample size, so it is not used for CV here.

**Read the caveat in the "Known limitation" cell below before trusting any AUROC number in this notebook.**
It is not optional context — it changes what the metrics mean.

## ⚠️ Known limitation: negative-sampling terrain confound (read before interpreting results)

The Phase 2 negative-sampling protocol requires negatives to sit on **stable terrain (slope < 5°)**,
anywhere in the 2.2M km² ROI. Checking the actual feature distributions:

- **Slope / TRI** separate the classes almost perfectly by construction (negatives ≈ 0–5°, positives
  mean ≈ 22°) — this is not learned susceptibility, it's the sampling cutoff itself.
- **Elevation** is bimodal for negatives (stable terrain sampled from both low-elevation plains *and*
  high-elevation plateaus/glaciated flats), while positives cluster at low-to-mid elevation
  (85% of positives sit below 2500m). This lets elevation act as a partial proxy for the same
  terrain-stability criterion even with slope/tri removed.
- **SAR backscatter (VH_post/VV_post)** also differs systematically between flat and steep terrain due
  to radar incidence-angle geometry, independent of any rainfall signal — another partial proxy.

**What this means:** AUROC ≈ 0.95–0.98 in this notebook substantially reflects "can the model tell
flat/stable terrain from steep/unstable terrain" rather than "given comparably unstable terrain, can the
model tell which locations are more rainfall-susceptible." The second question is the one that actually
matters for a susceptibility map, and this dataset's negative-sampling design doesn't cleanly test it.

**What we do about it here (per decision: report both):**
- Report metrics for the **full feature set** and a **reduced set with slope/tri removed**, side by side.
- Report both explicitly as upper-bound estimates inflated by the sampling design, not as final
  validated skill — the reduced-set number is a *partial* mitigation, not a clean fix, per the
  elevation/SAR argument above.
- **Recommended follow-up for Phase 2 (not done in this notebook):** redraw negatives as "hard negatives"
  — stable terrain matched in elevation band and proximity to the same steep-terrain context as the
  positives — rather than any slope<5° location anywhere in the ROI. That's the real fix; feature
  removal in Phase 4 can only partially compensate for a negative-sampling design issue.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from pyproj import Transformer
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import xgboost as xgb
import joblib

df = pd.read_csv("phase3_feature_table.csv")
df["event_date"] = pd.to_datetime(df["event_date"])
print(f"Loaded {len(df)} rows, {df['label'].sum()} positive / {(df['label']==0).sum()} negative")


## 1. Feature sets

Drop `VH_pre`/`VV_pre` (exact linear components of `dVH`/`dVV`, causes rank-deficient VIF — see Phase 3).
Build both a full and a slope/tri-reduced feature set per the caveat above.

In [ ]:
non_feature_cols = ["latitude", "longitude", "event_date", "Location Accuracy",
                     "Landslide Trigger", "Imported Event Source Catalog", "label", "block_id"]
redundant_sar_cols = ["VH_pre", "VV_pre"]  # exact linear components of dVH/dVV -- see Phase 3 fix

full_feature_cols = [c for c in df.columns if c not in non_feature_cols + redundant_sar_cols]
reduced_feature_cols = [c for c in full_feature_cols if c not in ["slope", "tri"]]

print(f"Full feature set ({len(full_feature_cols)}):", full_feature_cols)
print(f"\nReduced feature set ({len(reduced_feature_cols)}):", reduced_feature_cols)

y = df["label"].copy()


## 2. Spatial Cluster K-Fold setup

Per the Phase 2/3 semivariogram finding (residual autocorrelation range ~438km, far exceeding the
`block_id` grid width), we use **KMeans on azimuthal-equidistant-projected coordinates** for both the
outer evaluation folds and the inner hyperparameter-search folds, re-clustering on each training subset
so the inner search never sees the outer test cluster.

In [ ]:
lat0, lon0 = df["latitude"].mean(), df["longitude"].mean()
transformer = Transformer.from_crs(
    "EPSG:4326",
    f"+proj=aeqd +lat_0={lat0} +lon_0={lon0} +datum=WGS84 +units=m",
    always_xy=True,
)
x_m, y_m = transformer.transform(df["longitude"].values, df["latitude"].values)
coords_m = np.column_stack([x_m, y_m])

N_OUTER_FOLDS = 5
N_INNER_FOLDS = 4

outer_fold_id = KMeans(n_clusters=N_OUTER_FOLDS, random_state=42, n_init=10).fit_predict(coords_m)
print("Outer spatial fold sizes:", np.bincount(outer_fold_id))


## 3. Train + evaluate (both feature sets, both models)

Hyperparameter search happens *inside* each outer fold's training data only, using inner spatial groups -- never touching that fold's held-out test points.

In [ ]:
RF_PARAM_DIST = {
    "clf__n_estimators": [200, 400, 600],
    "clf__max_depth": [None, 4, 8, 12],
    "clf__min_samples_leaf": [1, 2, 4],
    "clf__max_features": ["sqrt", "log2"],
}
XGB_PARAM_DIST = {
    "n_estimators": [200, 400, 600],
    "max_depth": [3, 4, 6],
    "learning_rate": [0.03, 0.05, 0.1],
    "subsample": [0.7, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.9, 1.0],
}

def run_spatial_cv(feature_cols, label):
    X = df[feature_cols]
    oof_rf, oof_xgb = np.zeros(len(df)), np.zeros(len(df))
    fold_metrics = []
    fold_models = {}

    for fold in range(N_OUTER_FOLDS):
        test_mask = outer_fold_id == fold
        train_mask = ~test_mask
        X_train, X_test = X[train_mask], X[test_mask]
        y_train, y_test = y[train_mask], y[test_mask]

        # Inner spatial groups, recomputed on this fold's training subset only
        inner_groups = KMeans(n_clusters=N_INNER_FOLDS, random_state=42, n_init=10).fit_predict(coords_m[train_mask])
        inner_cv = list(GroupKFold(N_INNER_FOLDS).split(X_train, y_train, groups=inner_groups))

        # Random Forest -- needs imputation, sklearn RF can't take NaN directly
        rf_pipe = Pipeline([("impute", SimpleImputer(strategy="median")),
                             ("clf", RandomForestClassifier(class_weight="balanced", random_state=42))])
        rf_search = RandomizedSearchCV(rf_pipe, RF_PARAM_DIST, n_iter=8, cv=inner_cv,
                                        scoring="roc_auc", random_state=42, n_jobs=-1).fit(X_train, y_train)
        oof_rf[test_mask] = rf_search.best_estimator_.predict_proba(X_test)[:, 1]

        # XGBoost -- native NaN handling, no imputation needed
        scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
        xgb_clf = xgb.XGBClassifier(eval_metric="logloss", scale_pos_weight=scale_pos_weight,
                                     random_state=42, tree_method="hist")
        xgb_search = RandomizedSearchCV(xgb_clf, XGB_PARAM_DIST, n_iter=8, cv=inner_cv,
                                         scoring="roc_auc", random_state=42, n_jobs=-1).fit(X_train, y_train)
        oof_xgb[test_mask] = xgb_search.best_estimator_.predict_proba(X_test)[:, 1]

        fold_metrics.append({
            "fold": fold, "n_test": int(test_mask.sum()),
            "rf_auroc": roc_auc_score(y_test, oof_rf[test_mask]) if y_test.nunique() > 1 else np.nan,
            "xgb_auroc": roc_auc_score(y_test, oof_xgb[test_mask]) if y_test.nunique() > 1 else np.nan,
        })
        fold_models[fold] = {"rf": rf_search.best_estimator_, "xgb": xgb_search.best_estimator_}

    overall = {
        "feature_set": label, "n_features": len(feature_cols),
        "rf_auroc": roc_auc_score(y, oof_rf), "xgb_auroc": roc_auc_score(y, oof_xgb),
        "rf_prauc": average_precision_score(y, oof_rf), "xgb_prauc": average_precision_score(y, oof_xgb),
        "rf_f1": f1_score(y, oof_rf > 0.5), "xgb_f1": f1_score(y, oof_xgb > 0.5),
    }

    print(f"\n=== {label} ({len(feature_cols)} features) ===")
    print(pd.DataFrame(fold_metrics))
    print(f"OOF AUROC  -- RF: {overall['rf_auroc']:.4f}   XGB: {overall['xgb_auroc']:.4f}")
    print(f"OOF PR-AUC -- RF: {overall['rf_prauc']:.4f}   XGB: {overall['xgb_prauc']:.4f}")
    print(f"OOF F1@0.5 -- RF: {overall['rf_f1']:.4f}   XGB: {overall['xgb_f1']:.4f}")

    return oof_rf, oof_xgb, overall, fold_models

oof_rf_full, oof_xgb_full, metrics_full, models_full = run_spatial_cv(full_feature_cols, "FULL feature set (includes slope, tri)")
oof_rf_reduced, oof_xgb_reduced, metrics_reduced, models_reduced = run_spatial_cv(reduced_feature_cols, "REDUCED feature set (slope, tri removed)")


## 4. Side-by-side comparison

Remember: **both** numbers are inflated by the elevation/SAR terrain confound described above. The gap between them is a lower bound on how much of the "full" score is the slope/tri artifact specifically -- it does not mean the reduced score is clean.

In [ ]:
comparison = pd.DataFrame([metrics_full, metrics_reduced]).set_index("feature_set")
print(comparison.round(4))

comparison.to_csv("phase4_metrics_comparison.csv")
print("\nSaved phase4_metrics_comparison.csv")


## 5. Save out-of-fold predictions + models for Phase 5

OOF predictions (not in-sample fitted values) are what should feed SHAP and the susceptibility raster in Phase 5, since in-sample predictions would inherit whatever overfitting each model did on its own training fold.

In [ ]:
oof_df = pd.DataFrame({
    "latitude": df["latitude"], "longitude": df["longitude"], "label": y,
    "spatial_fold": outer_fold_id,
    "oof_rf_full": oof_rf_full, "oof_xgb_full": oof_xgb_full,
    "oof_rf_reduced": oof_rf_reduced, "oof_xgb_reduced": oof_xgb_reduced,
})
oof_df.to_csv("phase4_oof_predictions.csv", index=False)
print("Saved phase4_oof_predictions.csv:", oof_df.shape)

# Also fit one final model per (feature set x algorithm) on ALL 254 points, for Phase 5 SHAP +
# the production susceptibility raster. These are NOT used for the metrics above (which are OOF only).
final_models = {}
for label, cols in [("full", full_feature_cols), ("reduced", reduced_feature_cols)]:
    X_all = df[cols]
    rf_final = Pipeline([("impute", SimpleImputer(strategy="median")),
                          ("clf", RandomForestClassifier(n_estimators=500, class_weight="balanced", random_state=42))])
    rf_final.fit(X_all, y)
    spw = (y == 0).sum() / max((y == 1).sum(), 1)
    xgb_final = xgb.XGBClassifier(eval_metric="logloss", scale_pos_weight=spw, random_state=42,
                                   tree_method="hist", n_estimators=500, max_depth=5, learning_rate=0.05)
    xgb_final.fit(X_all, y)
    final_models[label] = {"rf": rf_final, "xgb": xgb_final, "feature_cols": cols}

joblib.dump(final_models, "phase4_final_models.joblib")
print("Saved phase4_final_models.joblib (fit on all 254 points, for Phase 5 SHAP + susceptibility raster)")
